# Stage 1 - Data Ingestion

In [1]:
import pandas as pd
from pathlib import Path
from IPython.display import display

deliveries_df = pd.read_csv('dataset/deliveries.csv')
matches_df = pd.read_csv('dataset/matches.csv')

print('deliveries_df shape:', deliveries_df.shape)
print(deliveries_df.columns.tolist())
print(deliveries_df.dtypes)

print('matches_df shape:', matches_df.shape)
print(matches_df.columns.tolist())
print(matches_df.dtypes)

deliveries_df shape: (260920, 17)
['match_id', 'inning', 'batting_team', 'bowling_team', 'over', 'ball', 'batter', 'bowler', 'non_striker', 'batsman_runs', 'extra_runs', 'total_runs', 'extras_type', 'is_wicket', 'player_dismissed', 'dismissal_kind', 'fielder']
match_id            int64
inning              int64
batting_team          str
bowling_team          str
over                int64
ball                int64
batter                str
bowler                str
non_striker           str
batsman_runs        int64
extra_runs          int64
total_runs          int64
extras_type           str
is_wicket           int64
player_dismissed      str
dismissal_kind        str
fielder               str
dtype: object
matches_df shape: (1095, 20)
['id', 'season', 'city', 'date', 'match_type', 'player_of_match', 'venue', 'team1', 'team2', 'toss_winner', 'toss_decision', 'winner', 'result', 'result_margin', 'target_runs', 'target_overs', 'super_over', 'method', 'umpire1', 'umpire2']
id             

In [ ]:
# Stage 2 - Data Cleaning & Validation

In [2]:
deliveries_numeric_cols = ['match_id', 'inning', 'over', 'ball', 'batsman_runs', 'extra_runs', 'total_runs', 'is_wicket']
matches_numeric_cols = ['id', 'result_margin', 'target_runs', 'target_overs']

for column in deliveries_numeric_cols:
    deliveries_df[column] = pd.to_numeric(deliveries_df[column], errors='coerce').astype('Int64')

for column in matches_numeric_cols:
    matches_df[column] = pd.to_numeric(matches_df[column], errors='coerce')

matches_df['id'] = matches_df['id'].round().astype('Int64')

for column in ['extras_type', 'player_dismissed', 'dismissal_kind', 'fielder']:
    deliveries_df[column] = deliveries_df[column].fillna('')

for column in ['city', 'player_of_match', 'venue', 'team1', 'team2', 'toss_winner', 'toss_decision', 'winner', 'result', 'method']:
    matches_df[column] = matches_df[column].fillna('')

print(deliveries_df.isna().sum()[deliveries_df.isna().sum() > 0])
print(matches_df.isna().sum()[matches_df.isna().sum() > 0])

missing_match_ids = sorted(set(deliveries_df['match_id']) - set(matches_df['id']))
if missing_match_ids:
    print(missing_match_ids[:10])
else:
    print('match IDs aligned')

print(deliveries_df.dtypes)
print(matches_df.dtypes)

Series([], dtype: int64)
result_margin    19
target_runs       3
target_overs      3
dtype: int64
match IDs aligned
match_id            Int64
inning              Int64
batting_team          str
bowling_team          str
over                Int64
ball                Int64
batter                str
bowler                str
non_striker           str
batsman_runs        Int64
extra_runs          Int64
total_runs          Int64
extras_type           str
is_wicket           Int64
player_dismissed      str
dismissal_kind        str
fielder               str
dtype: object
id                   Int64
season                 str
city                   str
date                   str
match_type             str
player_of_match        str
venue                  str
team1                  str
team2                  str
toss_winner            str
toss_decision          str
winner                 str
result                 str
result_margin      float64
target_runs        float64
target_overs       floa

# Stage 3 - Data Transformation

In [3]:
deliveries_df['total_runs_per_ball'] = deliveries_df['batsman_runs'].fillna(0) + deliveries_df['extra_runs'].fillna(0)
deliveries_df['over_number'] = deliveries_df['over'] + 1
matches_df['season'] = matches_df['season'].astype(str)

analysis_df = deliveries_df.merge(matches_df, left_on='match_id', right_on='id')
analysis_df.head()

,match_id,inning,batting_team,bowling_team,over,ball,batter,bowler,non_striker,batsman_runs,...,toss_decision,winner,result,result_margin,target_runs,target_overs,super_over,method,umpire1,umpire2
0,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,1,SC Ganguly,P Kumar,BB McCullum,0,...,field,Kolkata Knight Riders,runs,140.0,223.0,20.0,N,,Asad Rauf,RE Koertzen
1,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,2,BB McCullum,P Kumar,SC Ganguly,0,...,field,Kolkata Knight Riders,runs,140.0,223.0,20.0,N,,Asad Rauf,RE Koertzen
2,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,3,BB McCullum,P Kumar,SC Ganguly,0,...,field,Kolkata Knight Riders,runs,140.0,223.0,20.0,N,,Asad Rauf,RE Koertzen
3,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,4,BB McCullum,P Kumar,SC Ganguly,0,...,field,Kolkata Knight Riders,runs,140.0,223.0,20.0,N,,Asad Rauf,RE Koertzen
4,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,5,BB McCullum,P Kumar,SC Ganguly,0,...,field,Kolkata Knight Riders,runs,140.0,223.0,20.0,N,,Asad Rauf,RE Koertzen


# Stage 4 - Core Analysis

In [4]:
runs_per_match = (
    analysis_df.groupby(['match_id', 'season', 'team1', 'team2'], as_index=False)['total_runs']
    .sum()
    .sort_values('total_runs', ascending=False)
)

team_scores = (
    deliveries_df.groupby(['match_id', 'batting_team'], as_index=False)['total_runs']
    .sum()
    .sort_values(['match_id', 'total_runs'], ascending=[True, False])
)

display(runs_per_match.head(10), team_scores.head(10))

,match_id,season,team1,team2,total_runs
1053,1426268,2024,Sunrisers Hyderabad,Royal Challengers Bengaluru,549
1031,1422126,2024,Sunrisers Hyderabad,Mumbai Indians,523
1065,1426280,2024,Kolkata Knight Riders,Punjab Kings,523
1066,1426281,2024,Delhi Capitals,Mumbai Indians,504
146,419137,2009/10,Chennai Super Kings,Rajasthan Royals,469
1058,1426273,2024,Sunrisers Hyderabad,Delhi Capitals,465
679,1136604,2018,Kolkata Knight Riders,Kings XI Punjab,459
987,1359512,2023,Lucknow Super Giants,Punjab Kings,458
626,1082641,2017,Mumbai Indians,Kings XI Punjab,453
791,1216527,2020/21,Kings XI Punjab,Rajasthan Royals,449


,match_id,batting_team,total_runs
0,335982,Kolkata Knight Riders,222
1,335982,Royal Challengers Bangalore,82
2,335983,Chennai Super Kings,240
3,335983,Kings XI Punjab,207
4,335984,Delhi Daredevils,132
5,335984,Rajasthan Royals,129
7,335985,Royal Challengers Bangalore,166
6,335985,Mumbai Indians,165
9,335986,Kolkata Knight Riders,112
8,335986,Deccan Chargers,110


In [5]:
top_batters = (
    deliveries_df.groupby('batter', as_index=False)['batsman_runs']
    .sum()
    .sort_values('batsman_runs', ascending=False)
    .head(10)
)

legal_deliveries = deliveries_df[~deliveries_df['extras_type'].fillna('').isin(['wides', 'noballs'])].copy()

strike_rate_runs = (
    legal_deliveries.groupby('batter', as_index=False)['batsman_runs']
    .sum()
    .rename(columns={'batsman_runs': 'runs'})
)
strike_rate_balls = (
    legal_deliveries.groupby('batter', as_index=False)
    .size()
    .rename(columns={'size': 'balls_faced'})
)
strike_rate = strike_rate_runs.merge(strike_rate_balls, on='batter', how='inner')
strike_rate['strike_rate'] = (strike_rate['runs'] / strike_rate['balls_faced'] * 100).round(2)
strike_rate = strike_rate.sort_values('strike_rate', ascending=False)

display(top_batters, strike_rate.head(10))

,batter,batsman_runs
631,V Kohli,8014
512,S Dhawan,6769
477,RG Sharma,6630
147,DA Warner,6567
546,SK Raina,5536
374,MS Dhoni,5243
30,AB de Villiers,5181
124,CH Gayle,4997
501,RV Uthappa,4954
282,KD Karthik,4843


,batter,runs,balls_faced,strike_rate
312,L Wood,9,3,300.0
97,B Stanlake,5,2,250.0
234,J Fraser-McGurk,324,139,233.09
461,R Sai Kishore,13,6,216.67
629,Umar Gul,39,19,205.26
497,RS Sodhi,4,2,200.0
465,R Shepherd,115,62,185.48
410,Naman Dhir,140,78,179.49
433,PD Salt,652,368,177.17
583,Shahid Afridi,81,46,176.09


In [6]:
bowler_runs = (
    deliveries_df.groupby('bowler', as_index=False)['total_runs']
    .sum()
    .rename(columns={'total_runs': 'runs_conceded'})
)
bowler_balls = (
    legal_deliveries.groupby('bowler', as_index=False)
    .size()
    .rename(columns={'size': 'balls_bowled'})
)
economy = bowler_runs.merge(bowler_balls, on='bowler', how='inner')
economy['overs_bowled'] = economy['balls_bowled'] / 6
economy['economy_rate'] = (economy['runs_conceded'] / economy['overs_bowled']).round(2)
economy = economy[economy['overs_bowled'] > 0].sort_values('economy_rate').head(10)

consistent_batters = (
    analysis_df.groupby('batter', as_index=False)
    .agg(total_runs=('batsman_runs', 'sum'), matches_played=('match_id', 'nunique'))
)
consistent_batters['average_runs_per_match'] = (
    consistent_batters['total_runs'] / consistent_batters['matches_played']
).round(2)
consistent_batters = consistent_batters[consistent_batters['matches_played'] >= 10].sort_values('average_runs_per_match', ascending=False)

display(economy, consistent_batters.head(10))

,bowler,runs_conceded,balls_bowled,overs_bowled,economy_rate
24,AC Gilchrist,0,1,0.166667,0.0
364,R Ravindra,7,12,2.000000,3.5
317,NB Singh,18,24,4.000000,4.5
460,Sachin Baby,8,10,1.666667,4.8
38,AM Rahane,5,6,1.000000,5.0
125,DJ Thornely,40,42,7.000000,5.71
260,M Manhas,42,42,7.000000,6.0
453,SS Mundhe,6,6,1.000000,6.0
245,LA Carseldine,6,6,1.000000,6.0
296,MW Short,25,24,4.000000,6.25


,batter,total_runs,matches_played,average_runs_per_match
170,DP Conway,924,22,42.0
96,B Sai Sudharsan,1034,25,41.36
289,KL Rahul,4689,122,38.43
319,LMP Simmons,1079,29,37.21
473,RD Gaikwad,2380,65,36.62
542,SE Marsh,2489,69,36.07
214,HM Amla,577,16,36.06
147,DA Warner,6567,184,35.69
124,CH Gayle,4997,141,35.44
365,ML Hayden,1107,32,34.59


In [7]:
highest_individual_score = (
    deliveries_df.groupby(['match_id', 'batter'], as_index=False)['batsman_runs']
    .sum()
    .sort_values('batsman_runs', ascending=False)
    .head(1)
)

boundary_deliveries = deliveries_df[deliveries_df['batsman_runs'].isin([4, 6])]

boundary_totals = (
    boundary_deliveries.groupby('batsman_runs')
    .size()
    .reset_index(name='count')
)

top_boundary_players = (
    boundary_deliveries.groupby('batter', as_index=False)
    .size()
    .sort_values('size', ascending=False)
    .head(10)
    .rename(columns={'size': 'boundaries'})
)

display(highest_individual_score, boundary_totals, top_boundary_players)

,match_id,batter,batsman_runs
5302,598027,CH Gayle,175


,batsman_runs,count
0,4,29850
1,6,13051


,batter,boundaries
532,V Kohli,981
437,S Dhawan,921
126,DA Warner,899
409,RG Sharma,880
109,CH Gayle,767
462,SK Raina,710
25,AB de Villiers,667
427,RV Uthappa,663
243,KD Karthik,627
323,MS Dhoni,615


In [8]:
boundary_runs = (
    boundary_deliveries.groupby('batter', as_index=False)['batsman_runs']
    .sum()
    .rename(columns={'batsman_runs': 'boundary_runs'})
)

total_runs_by_batter = (
    deliveries_df.groupby('batter', as_index=False)['batsman_runs']
    .sum()
    .rename(columns={'batsman_runs': 'total_runs'})
)

boundary_percentage = total_runs_by_batter.merge(boundary_runs, on='batter', how='left').fillna({'boundary_runs': 0})
boundary_percentage['boundary_percentage'] = (
    boundary_percentage['boundary_runs'] / boundary_percentage['total_runs'] * 100
).round(2)
boundary_percentage = boundary_percentage[boundary_percentage['total_runs'] > 0].sort_values('boundary_percentage', ascending=False)

dot_balls = deliveries_df[deliveries_df['total_runs'] == 0]
top_dot_ball_bowlers = (
    dot_balls.groupby('bowler', as_index=False)
    .size()
    .sort_values('size', ascending=False)
    .head(10)
    .rename(columns={'size': 'dot_balls'})
)

display(boundary_percentage.head(10), top_dot_ball_bowlers)

,batter,total_runs,boundary_runs,boundary_percentage
640,VRV Singh,4,4,100.0
99,BA Bhatt,6,6,100.0
497,RS Sodhi,4,4,100.0
325,Liton Das,4,4,100.0
18,A Tomar,4,4,100.0
419,P Chopra,8,8,100.0
198,GD McGrath,4,4,100.0
461,R Sai Kishore,13,12,92.31
234,J Fraser-McGurk,330,296,89.7
24,AA Kulkarni,9,8,88.89


,bowler,dot_balls
70,B Kumar,1632
436,SP Narine,1569
348,R Ashwin,1552
341,PP Chawla,1325
159,Harbhajan Singh,1263
188,JJ Bumrah,1228
366,RA Jadeja,1216
511,YS Chahal,1194
482,UT Yadav,1186
8,A Mishra,1185


In [9]:
runs_per_over = (
    deliveries_df.groupby('over_number', as_index=False)['total_runs']
    .mean()
    .rename(columns={'total_runs': 'average_runs'})
    .sort_values('over_number')
)

high_scoring_overs = runs_per_over.sort_values('average_runs', ascending=False).head(5)

powerplay_df = deliveries_df[deliveries_df['over_number'].between(1, 6)]
powerplay_total_runs = int(powerplay_df['total_runs'].sum())
powerplay_team_scores = (
    powerplay_df.groupby('batting_team', as_index=False)['total_runs']
    .sum()
    .sort_values('total_runs', ascending=False)
)

death_overs_df = deliveries_df[deliveries_df['over_number'].between(16, 20)]
death_overs_total_runs = int(death_overs_df['total_runs'].sum())
death_overs = (
    death_overs_df.groupby('batting_team', as_index=False)['total_runs']
    .sum()
    .sort_values('total_runs', ascending=False)
)

death_overs_batters = (
    death_overs_df.groupby('batter', as_index=False)['batsman_runs']
    .sum()
    .sort_values('batsman_runs', ascending=False)
    .head(10)
)

print('Powerplay total runs:', powerplay_total_runs)
print('Death overs total runs:', death_overs_total_runs)
display(high_scoring_overs, powerplay_team_scores.head(10), death_overs.head(10), death_overs_batters)

Powerplay total runs: 103217
Death overs total runs: 93884


,over_number,average_runs
19,20,1.776855
18,19,1.646896
17,18,1.587839
16,17,1.498778
15,16,1.434273


,batting_team,total_runs
10,Mumbai Indians,12225
8,Kolkata Knight Riders,11941
0,Chennai Super Kings,10991
16,Royal Challengers Bangalore,10919
13,Rajasthan Royals,10226
6,Kings XI Punjab,8954
18,Sunrisers Hyderabad,8937
3,Delhi Daredevils,7360
2,Delhi Capitals,4709
1,Deccan Chargers,3417


,batting_team,total_runs
10,Mumbai Indians,11889
0,Chennai Super Kings,11094
16,Royal Challengers Bangalore,10514
8,Kolkata Knight Riders,10123
13,Rajasthan Royals,9133
6,Kings XI Punjab,7875
18,Sunrisers Hyderabad,7672
3,Delhi Daredevils,6272
2,Delhi Capitals,3907
1,Deccan Chargers,3133


,batter,batsman_runs
337,MS Dhoni,3292
250,KA Pollard,2032
255,KD Karthik,1904
27,AB de Villiers,1868
425,RA Jadeja,1680
431,RG Sharma,1513
567,V Kohli,1469
34,AD Russell,1324
130,DA Miller,1298
190,HH Pandya,1290


In [10]:
inning_distribution = (
    deliveries_df.groupby('inning', as_index=False)['total_runs']
    .agg(total_runs='sum', average_runs='mean')
)

match_team_runs = (
    deliveries_df.groupby(['match_id', 'batting_team'], as_index=False)['total_runs']
    .sum()
)

toss_compare = match_team_runs.merge(
    matches_df[['id', 'toss_winner']],
    left_on='match_id',
    right_on='id',
    how='inner'
)
toss_compare['team_type'] = toss_compare['batting_team'].eq(toss_compare['toss_winner']).map({True: 'toss_winner', False: 'opponent'})
toss_impact = (
    toss_compare.groupby('team_type', as_index=False)['total_runs']
    .mean()
    .rename(columns={'total_runs': 'average_runs'})
)

player_match_scores = (
    deliveries_df.groupby(['match_id', 'batter'], as_index=False)['batsman_runs']
    .sum()
    .sort_values(['match_id', 'batsman_runs'], ascending=[True, False])
    .drop_duplicates('match_id')
    .rename(columns={'batter': 'top_scorer', 'batsman_runs': 'top_runs'})
)

player_of_match_contribution = matches_df[['id', 'player_of_match']].merge(
    player_match_scores,
    left_on='id',
    right_on='match_id',
    how='inner'
)
player_of_match_contribution['pom_is_top_scorer'] = (
    player_of_match_contribution['player_of_match'] == player_of_match_contribution['top_scorer']
)

player_of_match_summary = pd.DataFrame(
    {
        'matches': [len(player_of_match_contribution)],
        'matches_where_pom_is_top_scorer': [int(player_of_match_contribution['pom_is_top_scorer'].sum())]
    }
)
player_of_match_summary['percentage'] = (
    player_of_match_summary['matches_where_pom_is_top_scorer'] / player_of_match_summary['matches'] * 100
).round(2)

display(inning_distribution, toss_impact, player_of_match_summary)

,inning,total_runs,average_runs
0,1,181274,1.342591
1,2,166196,1.321733
2,3,137,1.779221
3,4,123,1.708333
4,5,11,1.375
5,6,15,3.75


,team_type,average_runs
0,opponent,161.35192
1,toss_winner,156.666972


,matches,matches_where_pom_is_top_scorer,percentage
0,1095,496,45.3


In [13]:
match_runs_for_venue = runs_per_match.copy()
if 'runs' in match_runs_for_venue.columns:
    match_runs_for_venue = match_runs_for_venue.rename(columns={'runs': 'total_runs'})

match_totals = match_runs_for_venue.merge(
    matches_df[['id', 'venue', 'city', 'winner']],
    left_on='match_id',
    right_on='id',
    how='inner'
)

venue_analysis = (
    match_totals.groupby('venue', as_index=False)
    .agg(total_matches=('match_id', 'nunique'), average_runs=('total_runs', 'mean'))
    .sort_values('average_runs', ascending=False)
)

city_analysis = (
    match_totals.groupby('city', as_index=False)['total_runs']
    .mean()
    .rename(columns={'total_runs': 'average_runs'})
    .sort_values('average_runs', ascending=False)
)

season_runs = (
    match_totals.groupby('season', as_index=False)['total_runs']
    .sum()
    .sort_values('season')
)
season_runs['change'] = season_runs['total_runs'].diff()

inning_teams = (
    deliveries_df.groupby(['match_id', 'inning'], as_index=False)['batting_team']
    .first()
)

inning_runs = (
    deliveries_df.groupby(['match_id', 'inning'], as_index=False)['total_runs']
    .sum()
)

match_innings = inning_runs.merge(inning_teams, on=['match_id', 'inning'], how='inner')

first_innings = match_innings[match_innings['inning'] == 1].rename(columns={'batting_team': 'first_innings_team', 'total_runs': 'first_innings_runs'})
second_innings = match_innings[match_innings['inning'] == 2].rename(columns={'batting_team': 'second_innings_team', 'total_runs': 'second_innings_runs'})

winning_team_analysis = first_innings.merge(
    second_innings,
    on='match_id',
    how='inner'
).merge(
    matches_df[['id', 'winner']],
    left_on='match_id',
    right_on='id',
    how='inner'
)

winning_team_analysis['predicted_winner'] = winning_team_analysis.apply(
    lambda row: row['first_innings_team']
    if row['first_innings_runs'] > row['second_innings_runs']
    else row['second_innings_team']
    if row['second_innings_runs'] > row['first_innings_runs']
    else 'Tie',
    axis=1
)

winning_team_analysis['prediction_matches_actual'] = (
    winning_team_analysis['predicted_winner'] == winning_team_analysis['winner']
)

display(venue_analysis.head(10), city_analysis.head(10), season_runs, winning_team_analysis.head(10))

,venue,total_matches,average_runs
12,Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket St...,2,400.0
1,"Arun Jaitley Stadium, Delhi",16,380.6875
15,"Eden Gardens, Kolkata",16,380.3125
24,"M Chinnaswamy Stadium, Bengaluru",14,380.142857
19,"Himachal Pradesh Cricket Association Stadium, ...",4,378.75
39,"Punjab Cricket Association IS Bindra Stadium, ...",5,371.2
43,"Rajiv Gandhi International Stadium, Uppal, Hyd...",13,364.846154
5,Brabourne Stadium,10,348.1
37,Punjab Cricket Association IS Bindra Stadium,10,347.6
56,"Wankhede Stadium, Mumbai",45,346.377778


,city,average_runs
4,Bengaluru,360.310345
16,Guwahati,339.666667
12,Dharamsala,339.384615
26,Mohali,335.0
33,Rajkot,333.3
2,Ahmedabad,331.027778
27,Mumbai,328.421965
8,Chandigarh,325.639344
10,Cuttack,325.428571
21,Kanpur,324.5


,season,total_runs,change
0,2007/08,17937,<NA>
1,2009,16353,-1584
2,2009/10,18883,2530
3,2011,21154,2271
4,2012,22453,1299
5,2013,22602,149
6,2014,18931,-3671
7,2015,18353,-578
8,2016,18862,509
9,2017,18786,-76


,match_id,inning_x,first_innings_runs,first_innings_team,inning_y,second_innings_runs,second_innings_team,id,winner,predicted_winner,prediction_matches_actual
0,335982,1,222,Kolkata Knight Riders,2,82,Royal Challengers Bangalore,335982,Kolkata Knight Riders,Kolkata Knight Riders,True
1,335983,1,240,Chennai Super Kings,2,207,Kings XI Punjab,335983,Chennai Super Kings,Chennai Super Kings,True
2,335984,1,129,Rajasthan Royals,2,132,Delhi Daredevils,335984,Delhi Daredevils,Delhi Daredevils,True
3,335985,1,165,Mumbai Indians,2,166,Royal Challengers Bangalore,335985,Royal Challengers Bangalore,Royal Challengers Bangalore,True
4,335986,1,110,Deccan Chargers,2,112,Kolkata Knight Riders,335986,Kolkata Knight Riders,Kolkata Knight Riders,True
5,335987,1,166,Kings XI Punjab,2,168,Rajasthan Royals,335987,Rajasthan Royals,Rajasthan Royals,True
6,335988,1,142,Deccan Chargers,2,143,Delhi Daredevils,335988,Delhi Daredevils,Delhi Daredevils,True
7,335989,1,208,Chennai Super Kings,2,202,Mumbai Indians,335989,Chennai Super Kings,Chennai Super Kings,True
8,335990,1,214,Deccan Chargers,2,217,Rajasthan Royals,335990,Rajasthan Royals,Rajasthan Royals,True
9,335991,1,182,Kings XI Punjab,2,116,Mumbai Indians,335991,Kings XI Punjab,Kings XI Punjab,True


# Stage 5 - Derived Insights

In [14]:
most_consistent_batter = consistent_batters.iloc[0]['batter'] if not consistent_batters.empty else 'N/A'
best_death_over_team = death_overs.iloc[0]['batting_team'] if not death_overs.empty else 'N/A'
high_scoring_venue = venue_analysis.iloc[0]['venue'] if not venue_analysis.empty else 'N/A'

print('Most consistent batter:', most_consistent_batter)
print('Best death-over team:', best_death_over_team)
print('High-scoring venue:', high_scoring_venue)

Most consistent batter: DP Conway
Best death-over team: Mumbai Indians
High-scoring venue: Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket Stadium, Visakhapatnam


# Stage 6 - Reporting

In [15]:
runs_per_match = runs_per_match.rename(columns={'total_runs': 'runs'}).sort_values('runs', ascending=False)
top_batters = top_batters.rename(columns={'batsman_runs': 'runs'}).sort_values('runs', ascending=False)
strike_rate = strike_rate[['batter', 'runs', 'balls_faced', 'strike_rate']].sort_values('strike_rate', ascending=False)
economy = economy[['bowler', 'runs_conceded', 'balls_bowled', 'overs_bowled', 'economy_rate']].sort_values('economy_rate')
team_scores = team_scores.rename(columns={'total_runs': 'runs'}).sort_values(['match_id', 'runs'], ascending=[True, False])
death_overs = death_overs.rename(columns={'total_runs': 'runs'}).sort_values('runs', ascending=False)
consistent_batters = consistent_batters.rename(columns={'total_runs': 'runs'}).sort_values('average_runs_per_match', ascending=False)
boundary_percentage = boundary_percentage.rename(columns={'total_runs': 'runs'})[['batter', 'runs', 'boundary_runs', 'boundary_percentage']].sort_values('boundary_percentage', ascending=False)
venue_analysis = venue_analysis.rename(columns={'total_matches': 'matches'}).sort_values('average_runs', ascending=False)
city_analysis = city_analysis.sort_values('average_runs', ascending=False)
season_runs = season_runs.sort_values('season')
powerplay_team_scores = powerplay_team_scores.rename(columns={'total_runs': 'runs'}).sort_values('runs', ascending=False)
death_overs_batters = death_overs_batters.rename(columns={'batsman_runs': 'runs'}).sort_values('runs', ascending=False)
high_scoring_overs = high_scoring_overs.sort_values('average_runs', ascending=False)
top_boundary_players = top_boundary_players.sort_values('boundaries', ascending=False)
top_dot_ball_bowlers = top_dot_ball_bowlers.sort_values('dot_balls', ascending=False)
inning_distribution = inning_distribution.sort_values('inning')
toss_impact = toss_impact.sort_values('average_runs', ascending=False)
player_of_match_summary = player_of_match_summary[['matches', 'matches_where_pom_is_top_scorer', 'percentage']]
winning_team_analysis = winning_team_analysis[['match_id', 'first_innings_team', 'second_innings_team', 'first_innings_runs', 'second_innings_runs', 'predicted_winner', 'winner', 'prediction_matches_actual']]

display(runs_per_match.head(10), top_batters, strike_rate.head(10), economy, team_scores.head(10), death_overs.head(10))

,match_id,season,team1,team2,runs
1053,1426268,2024,Sunrisers Hyderabad,Royal Challengers Bengaluru,549
1031,1422126,2024,Sunrisers Hyderabad,Mumbai Indians,523
1065,1426280,2024,Kolkata Knight Riders,Punjab Kings,523
1066,1426281,2024,Delhi Capitals,Mumbai Indians,504
146,419137,2009/10,Chennai Super Kings,Rajasthan Royals,469
1058,1426273,2024,Sunrisers Hyderabad,Delhi Capitals,465
679,1136604,2018,Kolkata Knight Riders,Kings XI Punjab,459
987,1359512,2023,Lucknow Super Giants,Punjab Kings,458
626,1082641,2017,Mumbai Indians,Kings XI Punjab,453
791,1216527,2020/21,Kings XI Punjab,Rajasthan Royals,449


,batter,runs
631,V Kohli,8014
512,S Dhawan,6769
477,RG Sharma,6630
147,DA Warner,6567
546,SK Raina,5536
374,MS Dhoni,5243
30,AB de Villiers,5181
124,CH Gayle,4997
501,RV Uthappa,4954
282,KD Karthik,4843


,batter,runs,balls_faced,strike_rate
312,L Wood,9,3,300.0
97,B Stanlake,5,2,250.0
234,J Fraser-McGurk,324,139,233.09
461,R Sai Kishore,13,6,216.67
629,Umar Gul,39,19,205.26
497,RS Sodhi,4,2,200.0
465,R Shepherd,115,62,185.48
410,Naman Dhir,140,78,179.49
433,PD Salt,652,368,177.17
583,Shahid Afridi,81,46,176.09


,bowler,runs_conceded,balls_bowled,overs_bowled,economy_rate
24,AC Gilchrist,0,1,0.166667,0.0
364,R Ravindra,7,12,2.000000,3.5
317,NB Singh,18,24,4.000000,4.5
460,Sachin Baby,8,10,1.666667,4.8
38,AM Rahane,5,6,1.000000,5.0
125,DJ Thornely,40,42,7.000000,5.71
260,M Manhas,42,42,7.000000,6.0
453,SS Mundhe,6,6,1.000000,6.0
245,LA Carseldine,6,6,1.000000,6.0
296,MW Short,25,24,4.000000,6.25


,match_id,batting_team,runs
0,335982,Kolkata Knight Riders,222
1,335982,Royal Challengers Bangalore,82
2,335983,Chennai Super Kings,240
3,335983,Kings XI Punjab,207
4,335984,Delhi Daredevils,132
5,335984,Rajasthan Royals,129
7,335985,Royal Challengers Bangalore,166
6,335985,Mumbai Indians,165
9,335986,Kolkata Knight Riders,112
8,335986,Deccan Chargers,110


,batting_team,runs
10,Mumbai Indians,11889
0,Chennai Super Kings,11094
16,Royal Challengers Bangalore,10514
8,Kolkata Knight Riders,10123
13,Rajasthan Royals,9133
6,Kings XI Punjab,7875
18,Sunrisers Hyderabad,7672
3,Delhi Daredevils,6272
2,Delhi Capitals,3907
1,Deccan Chargers,3133


# Stage 7 - Data Export

In [16]:
output_dir = Path('output')
output_dir.mkdir(exist_ok=True)

runs_per_match.to_csv(output_dir / 'runs_per_match.csv', index=False)
top_batters.to_csv(output_dir / 'top_batters.csv', index=False)
strike_rate.to_csv(output_dir / 'strike_rate.csv', index=False)
economy.to_csv(output_dir / 'economy.csv', index=False)
team_scores.to_csv(output_dir / 'team_scores.csv', index=False)
death_overs.to_csv(output_dir / 'death_overs.csv', index=False)

with pd.ExcelWriter(output_dir / 'ipl_analysis.xlsx') as writer:
    runs_per_match.to_excel(writer, sheet_name='Runs Per Match', index=False)
    top_batters.to_excel(writer, sheet_name='Top Batters', index=False)
    strike_rate.to_excel(writer, sheet_name='Strike Rate', index=False)
    economy.to_excel(writer, sheet_name='Economy', index=False)
    team_scores.to_excel(writer, sheet_name='Team Scores', index=False)
    death_overs.to_excel(writer, sheet_name='Death Overs', index=False)
    consistent_batters.to_excel(writer, sheet_name='Consistent Batters', index=False)
    boundary_percentage.to_excel(writer, sheet_name='Boundary Percentage', index=False)
    powerplay_team_scores.to_excel(writer, sheet_name='Powerplay Teams', index=False)
    death_overs_batters.to_excel(writer, sheet_name='Death Batters', index=False)
    high_scoring_overs.to_excel(writer, sheet_name='High Scoring Overs', index=False)
    boundary_totals.to_excel(writer, sheet_name='Boundary Totals', index=False)
    top_boundary_players.to_excel(writer, sheet_name='Boundary Leaders', index=False)
    top_dot_ball_bowlers.to_excel(writer, sheet_name='Dot Ball Bowlers', index=False)
    inning_distribution.to_excel(writer, sheet_name='Innings', index=False)
    toss_impact.to_excel(writer, sheet_name='Toss Impact', index=False)
    player_of_match_summary.to_excel(writer, sheet_name='Player of Match', index=False)
    venue_analysis.to_excel(writer, sheet_name='Venue Analysis', index=False)
    city_analysis.to_excel(writer, sheet_name='City Analysis', index=False)
    season_runs.to_excel(writer, sheet_name='Season Runs', index=False)
    winning_team_analysis.to_excel(writer, sheet_name='Winning Teams', index=False)

print('Export complete')

Export complete
